# CNN Image Classification - Cats vs Dogs vs Gungans
3-class classification with data augmentation. Gungans (Star Wars aliens) as a unique third class.

## 1. Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import zipfile
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


## 2. Prepare Dataset
`cat_dog_1.zip` is read from `../cat_dog_1.zip` (one level up). Gungan images download automatically on first run.

In [ ]:
import glob as _glob

# Reassemble cat_dog_1.zip from split parts if needed
if not os.path.exists('cat_dog_1.zip'):
    parts = sorted(_glob.glob('cat_dog_1.zip.part*'))
    if parts:
        print(f'Assembling {len(parts)} parts -> cat_dog_1.zip ...')
        with open('cat_dog_1.zip', 'wb') as out:
            for part in parts:
                with open(part, 'rb') as f:
                    out.write(f.read())
        print('Done.')

def find_zip(name):
    for base in ['.', '..', '../..']:
        hits = _glob.glob(f'{base}/**/{name}', recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(f'{name} not found')

zip_path = find_zip('cat_dog_1.zip')
print('Found zip:', zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('dataset_raw')

for root, dirs, files in os.walk('dataset_raw'):
    level = root.replace('dataset_raw', '').count(os.sep)
    print(' ' * level * 2 + os.path.basename(root) + '/')
    if level < 2:
        for f in files[:3]:
            print(' ' * (level+1) * 2 + f)

In [ ]:
CLASSES = ['cat', 'dog', 'gungan']

# clean up previous run to avoid stale/corrupt files
if os.path.exists('data'):
    shutil.rmtree('data')

for split in ['train', 'test']:
    for cls in CLASSES:
        os.makedirs(f'data/{split}/{cls}', exist_ok=True)

import glob

def copy_images(src_pattern, dst_folder):
    files = sorted(glob.glob(src_pattern, recursive=True))
    for f in files:
        shutil.copy(f, dst_folder)
    return len(files)

# zip structure: cat_dog_1/train/cat.XXXX.jpg  (no cats/ subfolder — class is in filename)
for cls_name in ['cat', 'dog']:
    n_train = copy_images(f'dataset_raw/**/train/{cls_name}.*.jpg', f'data/train/{cls_name}')
    n_test  = copy_images(f'dataset_raw/**/test/{cls_name}.*.jpg',  f'data/test/{cls_name}')
    if n_train == 0:  # fallback for training_set/test_set naming convention
        n_train = copy_images(f'dataset_raw/**/training_set/{cls_name}.*.jpg', f'data/train/{cls_name}')
        n_test  = copy_images(f'dataset_raw/**/test_set/{cls_name}.*.jpg',     f'data/test/{cls_name}')
    print(f'{cls_name}: {n_train} train, {n_test} test')

In [ ]:
import urllib.request, io
from PIL import Image as _PIL

GUNGAN_URLS = [
    'https://static.wikia.nocookie.net/starwars/images/5/5e/Aagungan.jpg/revision/latest?cb=20070820234434',
    'https://static.wikia.nocookie.net/starwars/images/b/b2/AckbarGunganPrisoners.jpg/revision/latest?cb=20201122213914',
    'https://static.wikia.nocookie.net/starwars/images/0/0a/Augie.png/revision/latest?cb=20210619085417',
    'https://static.wikia.nocookie.net/starwars/images/5/5f/BabyJarJar.jpg/revision/latest?cb=20221101161416',
    'https://static.wikia.nocookie.net/starwars/images/3/3b/BeenGCW.jpg/revision/latest?cb=20100825180207',
    'https://static.wikia.nocookie.net/starwars/images/b/b8/BigBlismo.png/revision/latest?cb=20221115175709',
    'https://static.wikia.nocookie.net/starwars/images/c/c8/DisIsNutsen-TPM.jpg/revision/latest?cb=20111008202231',
    'https://static.wikia.nocookie.net/starwars/images/6/62/DubbinSharr.jpg/revision/latest?cb=20131003010812',
    'https://static.wikia.nocookie.net/starwars/images/b/be/Fenton.jpg/revision/latest?cb=20160117183817',
    'https://static.wikia.nocookie.net/starwars/images/4/41/Flossin_Noz.jpg/revision/latest?cb=20111002174728',
    'https://static.wikia.nocookie.net/starwars/images/a/ab/1138Binks-TPM.png/revision/latest?cb=20140622174328',
    'https://static.wikia.nocookie.net/starwars/images/d/da/2163-Death-TPM.png/revision/latest?cb=20220321160126',
    'https://static.wikia.nocookie.net/starwars/images/c/c4/2163-Halt-TPM.png/revision/latest?cb=20220321155606',
    'https://static.wikia.nocookie.net/starwars/images/9/94/AnakincomplainstoJarJar.png/revision/latest?cb=20251007063601',
    'https://static.wikia.nocookie.net/starwars/images/e/ed/Binks22BBY.png/revision/latest?cb=20241220222900',
    'https://static.wikia.nocookie.net/starwars/images/5/5e/Binks_saves_Typho.png/revision/latest?cb=20140423202311',
    'https://static.wikia.nocookie.net/starwars/images/4/42/Binks_tank.png/revision/latest?cb=20120925000125',
    'https://static.wikia.nocookie.net/starwars/images/0/02/Bombad.jpg/revision/latest?cb=20090129203542',
    'https://static.wikia.nocookie.net/starwars/images/9/94/BombadBounty.jpg/revision/latest?cb=20210315164634',
    'https://static.wikia.nocookie.net/starwars/images/7/76/Bombad_Jedi.png/revision/latest?cb=20120921225535',
    'https://static.wikia.nocookie.net/starwars/images/2/25/BinksImprovSculpture-SL.png/revision/latest?cb=20130506225615',
    'https://static.wikia.nocookie.net/starwars/images/c/c8/BinksOrganaKatuunko-SL.png/revision/latest?cb=20221220050305',
    'https://static.wikia.nocookie.net/starwars/images/d/db/BinkssWoe.jpg/revision/latest?cb=20100918131610',
    'https://static.wikia.nocookie.net/starwars/images/5/5f/BlueShadowVirus.png/revision/latest?cb=20121002221140',
    'https://static.wikia.nocookie.net/starwars/images/e/e2/Bombad_bg.jpg/revision/latest?cb=20090122160344',
    'https://static.wikia.nocookie.net/starwars/images/5/54/Clones_jar_jar.png/revision/latest?cb=20120924235904',
    'https://static.wikia.nocookie.net/starwars/images/6/6c/DantumRoohd-SS.jpg/revision/latest?cb=20091017021754',
    'https://static.wikia.nocookie.net/starwars/images/a/ae/FrontLines-Gunganvictory.png/revision/latest?cb=20240511203132',
    'https://static.wikia.nocookie.net/starwars/images/9/9e/GalacticPals-Younglings.png/revision/latest?cb=20220426234347',
    'https://static.wikia.nocookie.net/starwars/images/6/69/BenbulLeedee.png/revision/latest?cb=20210226164326',
    'https://static.wikia.nocookie.net/starwars/images/2/23/BlueGungan-ELAC.png/revision/latest?cb=20121102022138',
    'https://static.wikia.nocookie.net/starwars/images/8/88/BootoLubble.png/revision/latest?cb=20210227183758',
    'https://static.wikia.nocookie.net/starwars/images/6/6c/BursaConceptArt.jpg/revision/latest?cb=20210924113934',
    'https://static.wikia.nocookie.net/starwars/images/8/88/Chase-the-Gungans.png/revision/latest?cb=20221120213120',
    'https://static.wikia.nocookie.net/starwars/images/1/1f/DaGunganBrigade-ELAC.png/revision/latest?cb=20121102015330',
    'https://static.wikia.nocookie.net/starwars/images/7/72/Epguide402.jpg/revision/latest?cb=20110917060912',
    'https://static.wikia.nocookie.net/starwars/images/f/f0/Epguide404.jpg/revision/latest?cb=20110923043906',
    'https://static.wikia.nocookie.net/starwars/images/3/33/Fambaa-FF47.png/revision/latest?cb=20230927033404',
    'https://static.wikia.nocookie.net/starwars/images/2/2d/DatabankElectropole.png/revision/latest?cb=20210515130525',
    'https://static.wikia.nocookie.net/starwars/images/7/7f/Fambaa-Horn.png/revision/latest?cb=20250919170948',
    'https://static.wikia.nocookie.net/starwars/images/c/c1/GB_Sacred_Place.png/revision/latest?cb=20121220192843',
]

preloaded = sorted(glob.glob('gungans/gungan_*.jpg'))
if not preloaded:
    for base in ['..', '../..']:
        preloaded = sorted(glob.glob(f'{base}/**/gungans/gungan_*.jpg', recursive=True))
        if preloaded:
            break

if preloaded:
    print(f'Using {len(preloaded)} pre-downloaded Gungan images')
    gungan_files = preloaded
else:
    os.makedirs('gungans', exist_ok=True)
    headers = {'User-Agent': 'Mozilla/5.0'}
    gungan_files = []
    for i, url in enumerate(GUNGAN_URLS):
        fname = f'gungans/gungan_{i:03d}.jpg'
        if os.path.exists(fname):
            gungan_files.append(fname)
            continue
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=15) as r:
                data = r.read()
            img = _PIL.open(io.BytesIO(data)).convert('RGB')
            img.save(fname, 'JPEG', quality=95)
            gungan_files.append(fname)
        except Exception as e:
            print(f'  Skip {i}: {e}')
    print(f'Downloaded {len(gungan_files)}/{len(GUNGAN_URLS)} Gungan images')

split_idx = int(len(gungan_files) * 0.85)
for f in gungan_files[:split_idx]:
    shutil.copy(f, 'data/train/gungan')
for f in gungan_files[split_idx:]:
    shutil.copy(f, 'data/test/gungan')
print(f'Gungans: {split_idx} train, {len(gungan_files)-split_idx} test')

In [5]:
# count images per class
for split in ['train', 'test']:
    print(f'--- {split} ---')
    for cls in CLASSES:
        n = len(os.listdir(f'data/{split}/{cls}'))
        print(f'  {cls}: {n}')

--- train ---
  cat: 0
  dog: 0
  gungan: 33
--- test ---
  cat: 0
  dog: 0
  gungan: 6


## 3. Data Generators

In [ ]:
IMG_SIZE = 64      # 64 instead of 128: 4x fewer pixels, significantly faster on CPU
BATCH_SIZE = 64    # larger batch = fewer steps per epoch

# heavy augmentation for train (especially helps with small gungan dataset)
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    brightness_range=[0.7, 1.3]
)

test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    'data/train',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_data = test_gen.flow_from_directory(
    'data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Class indices:', train_data.class_indices)

In [7]:
# preview augmented samples
sample_imgs, sample_labels = next(train_data)
label_idx = {v: k for k, v in train_data.class_indices.items()}

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, img, lbl in zip(axes.flatten(), sample_imgs[:10], sample_labels[:10]):
    ax.imshow(img)
    ax.set_title(label_idx[np.argmax(lbl)])
    ax.axis('off')
plt.suptitle('Augmented training samples')
plt.tight_layout()
plt.show()

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x12bcf5f80>

## 4. Build CNN Model

In [ ]:
def build_cnn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),

        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2),
        layers.Dropout(0.3),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn((IMG_SIZE, IMG_SIZE, 3), len(CLASSES))
model.summary()

## 5. Train

In [ ]:
# class weights to handle imbalance (fewer gungan images)
from sklearn.utils.class_weight import compute_class_weight

class_labels = train_data.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(class_labels),
    y=class_labels
)
class_weight_dict = dict(enumerate(class_weights))
print('Class weights:', class_weight_dict)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    train_data,
    epochs=50,
    validation_data=test_data,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## 6. Evaluate

In [ ]:
loss, acc = model.evaluate(test_data, verbose=0)
print(f'Test accuracy: {acc:.4f}')
print(f'Test loss:     {loss:.4f}')

In [ ]:
# classification report
test_data.reset()
y_pred_proba = model.predict(test_data, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_data.classes

print(classification_report(y_true, y_pred, target_names=CLASSES, labels=list(range(len(CLASSES)))))

## 7. Visualizations

In [ ]:
# training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred,
    display_labels=CLASSES,
    ax=ax, colorbar=False
)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# preview predictions on test images
test_gen_preview = ImageDataGenerator(rescale=1./255)
preview_data = test_gen_preview.flow_from_directory(
    'data/test',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=9,
    class_mode='categorical',
    shuffle=True
)

imgs, labels = next(preview_data)
preds = model.predict(imgs, verbose=0)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, img, true_lbl, pred_lbl in zip(axes.flatten(), imgs, labels, preds):
    true_cls = CLASSES[np.argmax(true_lbl)]
    pred_cls = CLASSES[np.argmax(pred_lbl)]
    conf = np.max(pred_lbl)
    color = 'green' if true_cls == pred_cls else 'red'
    ax.imshow(img)
    ax.set_title(f'True: {true_cls}\nPred: {pred_cls} ({conf:.2f})', color=color, fontsize=9)
    ax.axis('off')

plt.suptitle('Sample predictions (green = correct, red = wrong)')
plt.tight_layout()
plt.show()

In [ ]:
# save model
model.save('cat_dog_gungan_cnn.h5')
print('Model saved.')